# EWMA volatility per admissible pair

Visual sanity check for the per-pair EWMA sigma series computed by
`nysa_risk.volatility` against the price parquets in `data/underlying/`.

For each admissible `(collateral, borrowable)` pair we plot:

* the EWMA daily-sigma series (per-observation × √2, the "daily-scale"
  value the CLI prints), and
* the `sigma_stress` horizontal line (the configured stress quantile of
  the series).

Look for:

* Regime shifts (COVID-March-2020, FTX-Nov-2022, ...) as visible spikes.
* Weekend gaps on equity/crypto pairs — the alignment rule (equity
  calendar, crypto value = most recent close) means the Fri→Mon
  observation is measured against the crypto's actual weekend move,
  which usually shows up as sharper spikes on Monday than on other days
  in volatile crypto regimes.
* Short-history tickers (e.g. CRCL) with correspondingly short EWMA series.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from nysa_risk.config import load_universe
from nysa_risk.volatility import (
    DEFAULT_DATA_DIR,
    DAILY_SCALE,
    align_pair,
    ewma_volatility,
    load_prices,
    relative_log_returns,
    sigma_stress_value,
)

universe = load_universe()
lam = universe.calibration.ewma_lambda
q = universe.calibration.stress_quantile
print(f"lambda={lam}  stress_quantile={q}  price_history_years={universe.meta.price_history_years}")

In [ ]:
# Map every symbol (collateral or borrowable-as-collateral) to its yfinance ticker.
tickers = {c.symbol: c.underlying_ticker for c in universe.collaterals}
tickers.update({b.symbol: b.price_source for b in universe.borrowables})

pairs = universe.admissible_pairs()
print(f"{len(pairs)} admissible pairs")

In [ ]:
def compute_series(collateral_sym, borrowable_sym, data_dir=DEFAULT_DATA_DIR):
    c = load_prices(tickers[collateral_sym], Path(data_dir))
    b = load_prices(tickers[borrowable_sym], Path(data_dir))
    returns = relative_log_returns(c, b)
    sigma_series = ewma_volatility(returns, lam)
    stress = sigma_stress_value(sigma_series, q)
    return sigma_series, stress


def plot_pair(collateral_sym, borrowable_sym, ax=None):
    sigma_series, stress = compute_series(collateral_sym, borrowable_sym)
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 3))
    (sigma_series * DAILY_SCALE * 100).plot(ax=ax, lw=0.8)
    ax.axhline(stress * DAILY_SCALE * 100, color="red", ls="--", lw=0.8,
               label=f"sigma_stress ({int(q*100)}th pct) = {stress*DAILY_SCALE*100:.2f}%/day")
    ax.set_title(f"{collateral_sym} / {borrowable_sym}  (EWMA lambda={lam})")
    ax.set_ylabel("daily sigma (%)")
    ax.legend(loc="upper right", fontsize=8)
    ax.grid(True, alpha=0.3)
    return ax

In [ ]:
# Sample: one plot per pair (first 6 shown here to keep the notebook light).
sample = pairs[:6]
fig, axes = plt.subplots(len(sample), 1, figsize=(10, 2.5 * len(sample)), sharex=False)
if len(sample) == 1:
    axes = [axes]
for ax, (c_sym, b_sym) in zip(axes, sample):
    try:
        plot_pair(c_sym, b_sym, ax=ax)
    except FileNotFoundError as exc:
        ax.set_title(f"{c_sym} / {b_sym} — {exc}")
        ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Ranked table (matches the CLI output of `python -m nysa_risk.volatility`).
from nysa_risk.volatility import compute_all_pairs, format_table

results = compute_all_pairs(universe=universe)
print(format_table(results))